# Ch01-09: 대용량 데이터 처리


## 학습 목표

- Pandas vs Polars의 아키텍처 차이를 이해한다
- 청크 처리(chunk processing)로 메모리 한계를 극복한다
- 메모리 최적화 기법(다운캐스팅, 범주형 변환)을 적용한다
- 지연 평가(lazy evaluation)의 원리와 장점을 이해한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. Pandas vs Polars 아키텍처

**Pandas**: NumPy 기반, 단일 스레드, eager evaluation
- 메모리: 데이터의 2-5배 필요 (중간 결과 복사)

**Polars**: Arrow 기반, 멀티스레드, lazy evaluation
- 쿼리 최적화, 술어 밀어내기(predicate pushdown)
- 메모리 효율적: zero-copy, 참조 카운팅

### 2. 메모리 최적화

**다운캐스팅**: int64 → int8/16/32 (범위에 맞게)

| 타입 | 범위 | 메모리 |
|------|------|--------|
| int8 | -128~127 | 1 byte |
| int16 | -32768~32767 | 2 bytes |
| float32 | ±3.4e38 | 4 bytes |

**범주형 변환**: 반복 문자열 → category (메모리 절감 90%+)

### 3. 청크 처리

파일을 $k$ 행씩 읽어 순차 처리:
- 스트리밍 집계: 온라인 평균/분산 (Welford 알고리즘)
- 맵-리듀스 패턴

### 4. Welford의 온라인 알고리즘

$$M_k = M_{k-1} + \frac{x_k - M_{k-1}}{k}, \quad S_k = S_{k-1} + (x_k - M_{k-1})(x_k - M_k)$$

$$\text{Var} = S_n / (n-1)$$


## 구현 가이드

In [ ]:
import numpy as np, pandas as pd, time

# 대용량 합성 데이터 생성
np.random.seed(42)
n = 500_000
df = pd.DataFrame({
    'id': np.arange(n),
    'category': np.random.choice(['A','B','C','D','E']*200, n),
    'value1': np.random.normal(100, 20, n),
    'value2': np.random.uniform(0, 1000, n),
    'flag': np.random.randint(0, 2, n),
})

# 메모리 최적화
def optimize_memory(df):
    before = df.memory_usage(deep=True).sum()
    for col in df.select_dtypes(include=['int64']).columns:
        c_min, c_max = df[col].min(), df[col].max()
        if c_min >= 0 and c_max < 255: df[col] = df[col].astype('uint8')
        elif c_min > -128 and c_max < 127: df[col] = df[col].astype('int8')
        elif c_min > -32768 and c_max < 32767: df[col] = df[col].astype('int16')
        elif c_min > -2147483648 and c_max < 2147483647: df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].nunique() / len(df) < 0.5:
            df[col] = df[col].astype('category')
    after = df.memory_usage(deep=True).sum()
    print(f"메모리: {before/1e6:.1f}MB → {after/1e6:.1f}MB ({100*(1-after/before):.1f}% 절감)")
    return df

df_opt = optimize_memory(df.copy())


---
## 연습 문제

### 문제 1 [★]

1000만 행 합성 데이터에서 메모리 최적화(다운캐스팅, 범주형 변환)를 적용하고 최적화 전후 메모리/연산 시간을 비교하라.

In [ ]:
# TODO: 대용량 데이터 생성
# TODO: 최적화 전후 비교


### 문제 2 [★★]

Welford 온라인 알고리즘을 구현하고, 청크 처리로 메모리에 들어오지 않는 데이터의 평균/분산/백분위수를 정확하게 계산하라.

> **힌트**: 백분위수는 t-digest 또는 Q-digest 근사 알고리즘을 사용.

In [ ]:
class OnlineStats:
    # TODO: Welford algorithm
    pass


### 문제 3 [★★]

Pandas vs Polars 벤치마크: groupby, join, filter, sort 연산의 속도를 비교하라. (Polars 미설치 시 pandas 최적화 기법으로 대체)

In [ ]:
# TODO: 벤치마크 프레임워크


### 문제 4 [★★★]

스트리밍 GroupBy 집계기를 구현하라. 파일을 청크로 읽으면서 그룹별 평균/분산/카운트를 정확히 계산하고, 결과가 전체 데이터 로드와 동일함을 검증.

In [ ]:
class StreamingGroupBy:
    # TODO
    pass


---
## 참고 자료

- Polars documentation: https://pola.rs/
- Welford, B.P. (1962). Note on a Method for Calculating Corrected Sums of Squares.
- McKinney, W. (2017). Python for Data Analysis, 2nd ed.
